In [1]:
print("faris")

faris


In [ ]:
# za svaki bat i:                     # prolazimo kroz svakog šišmiša pojedinačno
#                                     # i = indeks jednog šišmiša u populaciji
 
#     beta = random(0, 1)             # slučajan broj između 0 i 1
#                                     # koristi se da frekvencija bude malo drugačija za svakog šišmiša
 
#     f[i] = f_min + (f_max - f_min) * beta
#                                     # f[i] = frekvencija i-tog šišmiša
#                                     # frekvencija određuje koliko jako mijenja svoje kretanje
#                                     # nije "pozicija", nego parametar koji utiče na pomak
 
#     v[i] = v[i] + (x[i] - best) * f[i]
#                                     # v[i] = brzina i-tog šišmiša
#                                     # brzina govori koliko i u kojem smjeru će se pomjeriti
#                                     # x[i] = trenutna pozicija tog šišmiša
#                                     # best = trenutno najbolje rješenje od svih šišmiša
#                                     # ovom formulom šišmiš koriguje svoju brzinu u odnosu na best
 
#     x_new = x[i] + v[i]
#                                     # x_new = nova kandidatska pozicija za tog jednog šišmiša
#                                     # znači: uzmemo staru poziciju i dodamo brzinu
#                                     # DA — ovo je nova pozicija JEDNOG šišmiša, ovog i-tog
 
#     ako random(0,1) > pulse_rate[i]:
#         x_new = best + epsilon * average_loudness
#                                     # ponekad šišmiš ne ide običnim pomakom
#                                     # nego skoči blizu trenutno najboljeg rješenja
#                                     # epsilon = mali slučajni broj / slučajan mali pomak
#                                     # average_loudness = prosječna glasnoća svih šišmiša
#                                     # ovo služi za lokalnu pretragu oko najboljeg rješenja
 
#     x_new = popravi_granice(x_new)
#                                     # ako je nova pozicija izašla van dozvoljenog opsega,
#                                     # vrati je unutar granica problema
 
#     fitness_new = objective(x_new)
#                                     # izračunaj koliko je dobra nova pozicija
#                                     # objective = funkcija koju minimiziraš ili maksimiziraš
 
#     ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
#         x[i] = x_new
#                                     # prihvati novu poziciju za tog šišmiša
 
#         fitness[i] = fitness_new
#                                     # zapamti novu vrijednost funkcije za tog šišmiša
 
#         loudness[i] = alpha * loudness[i]
#                                     # smanji glasnoću tog šišmiša
#                                     # što iteracije više idu, šišmiš postaje "mirniji"

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# ============================================================
# ALGORITAM ŠIŠMIŠA (Bat Algorithm) za optimizaciju SVM hiperparametara
# Yang, 2010 - Nature-Inspired Metaheuristic Algorithms
# ============================================================

# --- Parametri Bat Algoritma ---
N_BATS    = 20       # broj šišmiša (veličina populacije)
N_ITER    = 50       # broj iteracija
F_MIN     = 0.0      # minimalna frekvencija
F_MAX     = 2.0      # maksimalna frekvencija
ALPHA     = 0.9      # koeficijent smanjenja glasnoće
GAMMA     = 0.9      # koeficijent povećanja pulse_rate
A0        = 1.0      # početna glasnoća
R0        = 0.1      # početni pulse rate

# --- Prostor pretrage (kontinualni) ---
# Dimenzija 0: log10(C)  - raspon [log10(0.01), log10(100)] => [-2, 2]
# Dimenzija 1: tt_split  - raspon [0.1, 0.5]
BOUNDS = np.array([
    [-2.0, 2.0],   # log10(C)
    [0.10, 0.50],  # tt_split
])
DIM = len(BOUNDS)

# Fiksni parametri SVM (kernel i gamma)
KERNEL = "linear"
GAMMA  = "scale"

def clip_position(x, bounds):
    """Vrati poziciju unutar granica."""
    return np.clip(x, bounds[:, 0], bounds[:, 1])

def decode(x):
    """Pretvori kontinualnu poziciju u stvarne SVM parametre."""
    C        = 10 ** x[0]   # log skala
    tt_split = x[1]
    return C, tt_split

def objective(x, X_data, y_data, random_state=42):
    """Evaluacija: vraća F1 skor (viši = bolji => minimiziramo -F1)."""
    C, tt_split = decode(x)
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=tt_split, random_state=random_state
    )
    model = SVC(C=C, kernel=KERNEL, gamma=GAMMA)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return -f1_score(y_test, y_pred, average="weighted")  # negativan jer minimiziramo

# --- Inicijalizacija ---
np.random.seed(42)

positions  = np.random.uniform(BOUNDS[:, 0], BOUNDS[:, 1], (N_BATS, DIM))
velocities = np.zeros((N_BATS, DIM))
frequencies = np.zeros(N_BATS)
loudness   = np.full(N_BATS, A0)
pulse_rate = np.full(N_BATS, R0)

# Izračunaj početni fitness
fitness = np.array([objective(positions[i], X, y) for i in range(N_BATS)])

# Pronađi početno najboljeg šišmiša
best_idx  = np.argmin(fitness)
best_pos  = positions[best_idx].copy()
best_fit  = fitness[best_idx]

print(f"Početni best F1: {-best_fit:.4f} | C={10**best_pos[0]:.4f}, tt_split={best_pos[1]:.3f}")
print("-" * 60)

# --- Glavna petlja ---
avg_loudness = A0

for t in range(N_ITER):
    for i in range(N_BATS):

        # 1. Ažuriraj frekvenciju
        beta         = np.random.uniform(0, 1)
        frequencies[i] = F_MIN + (F_MAX - F_MIN) * beta

        # 2. Ažuriraj brzinu i poziciju
        velocities[i] = velocities[i] + (positions[i] - best_pos) * frequencies[i]
        x_new         = positions[i] + velocities[i]
        x_new         = clip_position(x_new, BOUNDS)

        # 3. Lokalna pretraga oko najboljeg (pulse rate)
        if np.random.rand() > pulse_rate[i]:
            epsilon = np.random.uniform(-1, 1, DIM)
            x_new   = best_pos + epsilon * avg_loudness

        x_new = clip_position(x_new, BOUNDS)

        # 4. Prihvati novu poziciju ako je bolja i glasnoća dopušta
        fit_new = objective(x_new, X, y)
        if fit_new < fitness[i] and np.random.rand() < loudness[i]:
            positions[i] = x_new
            fitness[i]   = fit_new

            # Ažuriraj glasnoću i pulse_rate
            loudness[i]   = ALPHA * loudness[i]
            pulse_rate[i] = R0 * (1 - np.exp(-GAMMA * (t + 1)))

        # 5. Ažuriraj globalno najboljeg
        if fitness[i] < best_fit:
            best_pos = positions[i].copy()
            best_fit = fitness[i]

    avg_loudness = np.mean(loudness)

    if (t + 1) % 10 == 0:
        C_best, tt_best = decode(best_pos)
        print(f"Iteracija {t+1:3d}/{N_ITER} | Best F1: {-best_fit:.4f} "
              f"| C={C_best:.4f}, tt_split={tt_best:.3f}")

# --- Rezultati ---
C_opt, tt_opt = decode(best_pos)
print("\n" + "=" * 60)
print(f"OPTIMALNI PARAMETRI (Bat Algorithm):")
print(f"  C         = {C_opt:.6f}")
print(f"  tt_split  = {tt_opt:.6f}")
print(f"  kernel    = {KERNEL}")
print(f"  gamma     = {GAMMA}")
print(f"  Best F1   = {-best_fit:.6f}")
print("=" * 60)

# Treniraj finalni model s optimalnim parametrima
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=tt_opt, random_state=42
)
final_model = SVC(C=C_opt, kernel=KERNEL, gamma=GAMMA)
final_model.fit(X_train, y_train)
y_pred_final = final_model.predict(X_test)
final_f1 = f1_score(y_test, y_pred_final, average="weighted")
print(f"\nFinalni model F1 na test skupu: {final_f1:.4f}")


In [7]:
OPCIJE = {
    # tt_split, random_state, C, kernel, gamma
    # "tt_split": [0.1, 0.2, 0.22, 0.25, 0.29, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5],
    "tt_split": [0.2, 0.22, 0.25, 0.29],
    # "tt_split": np.arange(0.1, 0.5, 0.0001).tolist(),
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}


# "tt_split": np.arange(0.32, 0.34, 0.0001).tolist(),

# SVM - Support Vector Machine - algoritam za klasifikaciju i regresiju

# RandomSearchCV - metoda za pronalaženje najboljih hiperparametara modela
# GridSearchCV - metoda za pronalaženje najboljih hiperparametara modela - isprobava sve kombinacije


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import random
import numpy as np

In [8]:
# Učitavanje podataka
df = pd.read_csv("iris.csv") 

X = df.drop('species', axis=1)
y = df['species']

In [9]:
X.head()

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [5]:
y.head()

0    setosa
1    setosa
2    setosa
3    setosa
4    setosa
Name: species, dtype: str

In [12]:
# Ovaj dio koda radi evaluaciju eksperimenata dati u varijabli OPCIJE.
# Za svaki eksperiment, dijeli podatke na trening i test skup, trenira SVM model sa zadanim hiperparametrima, i računa F1 score na test skupu.
# Koristeci GridSearchCV ili RandomSearchCV bi bilo efikasnije, ali ovaj kod demonstrira osnovni pristup evaluacije.

# Najbolji parametri: 
best_params = {
    "tt_split": 0.29,
    "random_state": 0,
    "C": 0.1,
    "kernel": "linear",
    "gamma": "scale"
}

best_score = 0
best_model = None

for tt_split in OPCIJE["tt_split"]:
    for random_state in OPCIJE["random_state"]:
        for C in OPCIJE["C"]:
            for kernel in OPCIJE["kernel"]:
                for gamma in OPCIJE["gamma"]:

                    # Podjela podataka na trening i test skup
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tt_split, random_state=random_state)

                    # Treniranje SVM modela
                    model = SVC(C=C, kernel=kernel, gamma=gamma)
                    model.fit(X_train, y_train)

                    # Predviđanje na test skupu
                    y_pred = model.predict(X_test)

                    # Računanje F1 score
                    score = f1_score(y_test, y_pred, average='weighted')
                    # ovdje moze ici bilo koja funkcija EVALUACIJE, npr. accuracy_score, precision_score, recall_score, itd.

                    # Ispis rezultata za trenutne parametre
                    print(f"tt_split: {tt_split}, random_state: {random_state}, C: {C}, kernel: {kernel}, gamma: {gamma} => F1 Score: {score}")
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            "tt_split": tt_split,
                            "random_state": random_state,
                            "C": C,
                            "kernel": kernel,
                            "gamma": gamma
                        }
                        best_model = model
                    
                    
            

tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: scale => F1 Score: 0.8380018674136321
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: auto => F1 Score: 0.9672820512820512
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf,

In [14]:
best_params, best_score


({'tt_split': 0.2,
  'random_state': 0,
  'C': 0.1,
  'kernel': 'linear',
  'gamma': 'scale'},
 1.0)

In [13]:
best_model


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [17]:
# RandomSearch Implementacija
from sklearn.model_selection import RandomizedSearchCV

# ValueError: Invalid parameter 'tt_split' for estimator SVC(). Valid parameters are: ['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'].

OPCIJE = {
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [True, False],
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    "decision_function_shape": ['ovo', 'ovr'],
    "degree": [3, 4, 5],
    "max_iter": [-1, 100, 200],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0, 1]
}

random_search = RandomizedSearchCV(estimator=SVC(), param_distributions=OPCIJE, n_iter=100, scoring='f1_weighted', random_state=0)
random_search.fit(X, y)
random_search.best_params_, random_search.best_score_



[LibSVM]*
optimization finished, #iter = 33
obj = -2.592990, rho = 0.039207
nSV = 47, nBSV = 43
Total nSV = 47
*
optimization finished, #iter = 24
obj = -2.566784, rho = 0.047265
nSV = 46, nBSV = 44
Total nSV = 46
*
optimization finished, #iter = 28
obj = -2.615216, rho = 0.040548
nSV = 47, nBSV = 44
Total nSV = 47
*
optimization finished, #iter = 25
obj = -2.523722, rho = 0.054816
nSV = 45, nBSV = 42
Total nSV = 45
*
optimization finished, #iter = 26
obj = -2.543544, rho = 0.068587
nSV = 46, nBSV = 43
Total nSV = 46
*
optimization finished, #iter = 27
obj = -2.716375, rho = -0.047019
nSV = 48, nBSV = 46
*
optimization finished, #iter = 20
obj = -1.610251, rho = -0.021724
nSV = 32, nBSV = 29
Total nSV = 32
*
optimization finished, #iter = 28
obj = -1.632391, rho = -0.038805
nSV = 33, nBSV = 29
Total nSV = 33
*
optimization finished, #iter = 16
obj = -1.614519, rho = -0.022762
nSV = 32, nBSV = 30
Total nSV = 32
*
optimization finished, #iter = 21
obj = -1.627161, rho = -0.030967
nSV = 3

/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = meth

*
optimization finished, #iter = 42
obj = -2.135316, rho = -0.090879
nSV = 13, nBSV = 6
Total nSV = 13
*
optimization finished, #iter = 26
obj = -1.844721, rho = 0.008719
nSV = 11, nBSV = 5
Total nSV = 11
*
optimization finished, #iter = 20
obj = -2.166311, rho = -0.048441
nSV = 11, nBSV = 6
Total nSV = 11
*
optimization finished, #iter = 36
obj = -2.035117, rho = -0.091072
nSV = 12, nBSV = 3
Total nSV = 12
*
optimization finished, #iter = 21
obj = -2.171364, rho = -0.073945
nSV = 11, nBSV = 6
Total nSV = 11
*
optimization finished, #iter = 21
obj = -2.171390, rho = 0.074190
nSV = 12, nBSV = 6
*
optimization finished, #iter = 31
obj = -1.730120, rho = -0.174056
nSV = 11, nBSV = 2
Total nSV = 11
*
optimization finished, #iter = 23
obj = -1.774519, rho = -0.145459
nSV = 10, nBSV = 2
Total nSV = 10
*
optimization finished, #iter = 32
obj = -1.756822, rho = -0.121355
nSV = 12, nBSV = 3
Total nSV = 12
*
optimization finished, #iter = 38
obj = -1.663445, rho = -0.168088
nSV = 12, nBSV = 2
To

/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=100).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=100).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=100).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.wa

*
optimization finished, #iter = 21
obj = -2.184294, rho = -0.150519
nSV = 12, nBSV = 6
Total nSV = 12
*
optimization finished, #iter = 42
obj = -1.903405, rho = -0.177588
nSV = 11, nBSV = 4
Total nSV = 11
*
optimization finished, #iter = 18
obj = -2.186771, rho = -0.177075
nSV = 11, nBSV = 6
Total nSV = 11
*
optimization finished, #iter = 31
obj = -1.972715, rho = -0.068382
nSV = 14, nBSV = 5
Total nSV = 14
*
optimization finished, #iter = 18
obj = -2.180982, rho = -0.150375
nSV = 11, nBSV = 6
Total nSV = 11
*
optimization finished, #iter = 30
obj = -2.195771, rho = 0.168645
nSV = 10, nBSV = 4
*
optimization finished, #iter = 25
obj = -1.734855, rho = -0.174108
nSV = 12, nBSV = 4
Total nSV = 12
*
optimization finished, #iter = 22
obj = -1.545872, rho = -0.194303
nSV = 11, nBSV = 3
Total nSV = 11
*
optimization finished, #iter = 22
obj = -1.732419, rho = -0.210947
nSV = 11, nBSV = 2
Total nSV = 11
*
optimization finished, #iter = 31
obj = -1.765167, rho = -0.201596
nSV = 13, nBSV = 3
T

...

*
optimization finished, #iter = 29
obj = -2.477935, rho = -0.100355
nSV = 7, nBSV = 0
Total nSV = 7
*
optimization finished, #iter = 31
obj = -1.954072, rho = -0.225347
nSV = 8, nBSV = 0
Total nSV = 8
*
optimization finished, #iter = 18
obj = -2.475338, rho = -0.101979
nSV = 6, nBSV = 0
Total nSV = 6
*
optimization finished, #iter = 26
obj = -2.476568, rho = -0.091259
nSV = 6, nBSV = 0
Total nSV = 6
*
optimization finished, #iter = 26
obj = -2.432706, rho = -0.124164
nSV = 6, nBSV = 0
Total nSV = 6
*
optimization finished, #iter = 24
obj = -2.477947, rho = 0.099263
nSV = 7, nBSV = 0
*
optimization finished, #iter = 22
obj = -1.803114, rho = -0.158433
nSV = 10, nBSV = 0
Total nSV = 10
*
optimization finished, #iter = 19
obj = -1.564810, rho = -0.175793
nSV = 7, nBSV = 0
Total nSV = 7
*
optimization finished, #iter = 23
obj = -1.791013, rho = -0.207507
nSV = 9, nBSV = 0
Total nSV = 9
*
optimization finished, #iter = 21
obj = -1.838525, rho = -0.182726
nSV = 9, nBSV = 0
Total nSV = 9
*
o

/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = meth

*
optimization finished, #iter = 21
obj = -2.487912, rho = -0.072889
nSV = 6, nBSV = 0
Total nSV = 6
*
optimization finished, #iter = 39
obj = -2.175244, rho = -0.038723
nSV = 7, nBSV = 0
Total nSV = 7
*
optimization finished, #iter = 22
obj = -2.491581, rho = -0.051714
nSV = 6, nBSV = 0
Total nSV = 6
*
optimization finished, #iter = 28
obj = -2.226448, rho = -0.126801
nSV = 6, nBSV = 0
Total nSV = 6
*
optimization finished, #iter = 40
obj = -2.492496, rho = -0.054447
nSV = 8, nBSV = 0
Total nSV = 8
*
optimization finished, #iter = 34
obj = -2.492496, rho = 0.054392
nSV = 8, nBSV = 0
*
optimization finished, #iter = 46
obj = -1.709771, rho = -0.106257
nSV = 11, nBSV = 0
Total nSV = 11
*
optimization finished, #iter = 30
obj = -1.906277, rho = -0.142572
nSV = 9, nBSV = 0
Total nSV = 9
*
optimization finished, #iter = 57
obj = -1.942948, rho = -0.166039
nSV = 11, nBSV = 0
Total nSV = 11
*
optimization finished, #iter = 35
obj = -1.827952, rho = -0.214770
nSV = 9, nBSV = 0
Total nSV = 9
*

/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/phd/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = meth

({'verbose': 0,
  'tol': 0.001,
  'shrinking': False,
  'random_state': 1,
  'probability': True,
  'max_iter': 100,
  'kernel': 'linear',
  'gamma': 'scale',
  'degree': 5,
  'decision_function_shape': 'ovr',
  'coef0': 0.0,
  'class_weight': None,
  'cache_size': 400,
  'break_ties': True,
  'C': 0.5},
 np.float64(0.9866332497911445))

In [ ]:
OPCIJE = {
    "random_state": np.arange(0, 50).tolist(),
    "C": np.arange(0.1, 10.1, 0.1).tolist(),
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [True, False],
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    "decision_function_shape": ['ovo', 'ovr'],
    "degree": [3, 4, 5],
    "max_iter": [-1, 100, 200],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0, 1]
    #sample
}

random_search = RandomizedSearchCV(estimator=SVC(), param_distributions=OPCIJE, n_iter=100, scoring='f1_weighted', random_state=0)
random_search.fit(X, y)
random_search.best_params_, random_search.best_score_